In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../../BIG_DATA/data/Patrimoine_Arbore_Nettoye.csv")

def preparation_donnees(df):
    # Variables clés : haut_tot et tronc_diam pour l'apprentissage, lat et long pour positionner les arbres sur les cartes
    colonnes_utiles = ['haut_tot', 'tronc_diam', 'lat', 'long']
    df_prepare = df[colonnes_utiles].copy().dropna()

    # La donnée est en cm dans le CSV, on passe en mètres pour la hauteur (plus parlant)
    # Le tronc_diam est en cm
    df_prepare['haut_tot'] = df_prepare['haut_tot'] / 100

    scaler = StandardScaler()
    features = ['haut_tot', 'tronc_diam']
    X_scaled = scaler.fit_transform(df_prepare[features])

    # print("Moyenne après scaling:", X_scaled.mean(axis=0)) # On attend une valeur très faible (=0)
    # print("Écart-type après scaling:", X_scaled.std(axis=0)) # On attend la valeur 1

    return df_prepare, X_scaled, scaler

df_prepare, X_scaled, scaler = preparation_donnees(df)

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import matplotlib.pyplot as plt

def evaluer_modeles(X):
    df_scores = []
    # Le minimum est deux clusters et on ne calculera pas au dessus de 10
    k_range = range(2, 11)

    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=1, n_init=10)
        labels = kmeans.fit_predict(X)

        df_scores.append({
            "n_clusters": k,
            "Silhouette (Higher is better)": silhouette_score(X, labels),
            "Calinski-Harabasz (Higher is better)": calinski_harabasz_score(X, labels),
            "Davies-Bouldin (Lower is better)": davies_bouldin_score(X, labels)
        })

    # Conversion en DataFrame
    df_res = pd.DataFrame(df_scores).set_index("n_clusters")

    # Visualisation
    axes = df_res.plot(subplots=True, layout=(1,3), figsize=(15,5), marker='o', sharex=True)

    plt.tight_layout()
    plt.show()

evaluer_modeles(X_scaled)

In [ ]:
import plotly.express as px

def generer_carte(k_choisi):
    kmeans = KMeans(n_clusters=k_choisi, random_state=1, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    df_prepare['cluster_id'] = clusters

    # Tri logique : on force l'ID 0 à être le plus petit en hauteur moyenne
    ordre = df_prepare.groupby('cluster_id')['haut_tot'].mean().sort_values().index
    mapping_tri = {old: str(new) for new, old in enumerate(ordre)}
    df_prepare['Categorie_ID'] = df_prepare['cluster_id'].map(mapping_tri)

    # Libellés selon le choix utilisateur
    noms_dynamiques = {i: f"Groupe {i}" for i in range(k_choisi)}

    if k_choisi == 2:
        noms_dynamiques = {0: "Petit", 1: "Grand"}
    elif k_choisi == 3:
        noms_dynamiques = {0: "Petit", 1: "Moyen", 2: "Grand"}

    df_prepare['Nom_Categorie'] = df_prepare['Categorie_ID'].astype(int).map(noms_dynamiques)

    liste_ordonnee = [noms_dynamiques[i] for i in range(k_choisi)]

    fig = px.scatter_map(
        df_prepare,
        lat="lat", lon="long",
        color="Nom_Categorie",
        size="haut_tot",
        hover_data=['haut_tot', 'tronc_diam'],
        category_orders={"Nom_Categorie": liste_ordonnee},
        map_style="open-street-map",
        zoom=12, height=600
    )
    fig.show()

try:
    nb = int(input("Nombre de catégories ? : "))
    generer_carte(nb)
except:
    print("Entrée invalide.")

In [ ]:
import joblib
from sklearn.cluster import KMeans

K_FINAL = 2
model_final = KMeans(n_clusters=K_FINAL, random_state=1, n_init=10)
clusters = model_final.fit_predict(X_scaled)

# Calcul des moyennes pour le mapping
df_tri = pd.DataFrame({'cluster_id': clusters, 'hauteur_m': df_prepare['haut_tot']})
moyennes = df_tri.groupby('cluster_id')['hauteur_m'].mean().sort_values()

ordre_ids = moyennes.index

# Gestion dynamique des noms
if len(ordre_ids) == 3:
    noms_labels = ["Petit", "Moyen", "Grand"]
elif len(ordre_ids) == 2:
    noms_labels = ["Petit", "Grand"]
else:
    # Pour k > 3, on génère des étiquettes numériques ou textuelles automatiques
    noms_labels = [f"Taille {i}" for i in range(len(ordre_ids))]

# Création du mapping sécurisée
mapping_noms = {int(cluster_id): noms_labels[i] for i, cluster_id in enumerate(ordre_ids)}
print(moyennes)

# Sauvegarde
joblib.dump(model_final, 'modele_arbres.pkl')
joblib.dump(scaler, 'scaler_arbres.pkl')
joblib.dump(mapping_noms, 'mapping_noms.pkl')

In [ ]:
import joblib
import pandas as pd

def diagnostic_arbre():
    model = joblib.load('modele_arbres.pkl')
    scaler = joblib.load('scaler_arbres.pkl')
    mapping = joblib.load('mapping_noms.pkl')

    print(f"Diagnostic : ({len(mapping)} catégories)")
    h = float(input("Hauteur souhaitée en MÈTRES (ex: 15) : "))
    d = float(input("Diamètre souhaité en CM (ex: 100) : "))

    entree = pd.DataFrame([[h, d]], columns=['haut_tot', 'tronc_diam'])
    scaled = scaler.transform(entree)

    res_id = int(model.predict(scaled)[0])
    nom_final = mapping.get(res_id, "Inconnu")

    print(f"\nCet arbre est classé : {nom_final}")

diagnostic_arbre()